<a href="https://colab.research.google.com/github/Lakshitha-Kavinda/cnn/blob/main/220316V_a02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Name:** Kavinda L.G.L.  
**Index number:** 220316V  
**GitHub:** https://github.com/Lakshitha-Kavinda/cnn.git

**AI tool acknowledgment:** I have used Claude to help with coding.

## Setup

In [1]:
import random, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

torch 2.11.0+cu130 | device: cuda
Tesla T4


### CIFAR-10: 45k train / 5k val, per-channel normalization with training statistics

In [2]:
CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

full = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
X_all = torch.from_numpy(full.data).permute(0, 3, 1, 2).float() / 255.0   # N,3,32,32 in [0,1]
y_all = torch.tensor(full.targets)

# fixed split: 45k train / 5k val
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(X_all), generator=g)
tr_idx, va_idx = perm[:45000], perm[45000:]

# per-channel statistics from the TRAINING split only
MEAN = X_all[tr_idx].mean(dim=(0, 2, 3), keepdim=True)
STD  = X_all[tr_idx].std(dim=(0, 2, 3), keepdim=True)
normalize = lambda x: (x - MEAN) / STD

X_train, y_train = normalize(X_all[tr_idx]).to(device), y_all[tr_idx].to(device)
X_val,   y_val   = normalize(X_all[va_idx]).to(device), y_all[va_idx].to(device)

def load_test():
    """Task 4 ONLY. Call once, on the final model."""
    t = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)
    X = torch.from_numpy(t.data).permute(0, 3, 1, 2).float() / 255.0
    return normalize(X).to(device), torch.tensor(t.targets).to(device)

print('train', tuple(X_train.shape), '| val', tuple(X_val.shape))
print('mean', MEAN.flatten().tolist(), '\nstd ', STD.flatten().tolist())

100%|██████████| 170M/170M [50:41<00:00, 56.1kB/s]


train (45000, 3, 32, 32) | val (5000, 3, 32, 32)
mean [0.49165481328964233, 0.4823332130908966, 0.4466792047023773] 
std  [0.24708116054534912, 0.2435086965560913, 0.2615806758403778]


## Task 01